In [2]:
# Ensure that dnq directory is updated to the latest version of code on GitHub
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd drive/MyDrive/Robotics/grokking-deep-rl-ide-version

/content/drive/MyDrive/Robotics/grokking-deep-rl-ide-version


In [4]:
# %cd grokking-deep-rl-ide-version
%cd dnq_ddnq_chp_9

/content/drive/MyDrive/Robotics/grokking-deep-rl-ide-version/dnq_ddnq_chp_9


In [42]:
!git pull origin main

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 442 bytes | 6.00 KiB/s, done.
From https://github.com/drenfr01/grokking-deep-rl-ide-version
 * branch            main       -> FETCH_HEAD
   8476c42..f2d4aa5  main       -> origin/main
Updating 8476c42..f2d4aa5
Fast-forward
 dnq_ddnq_chp_9/dnq/dnq.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


In [43]:
import warnings ; warnings.filterwarnings('ignore')
import os, sys

try:
    _nb_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _nb_dir = os.getcwd()
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

import time
import torch
import torch.optim as optim

import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from textwrap import wrap

import os.path
import os

LEAVE_PRINT_EVERY_N_SECS = 60
ERASE_LINE = '\x1b[2K'
EPS = 1e-6
BEEP = lambda: os.system("printf '\a'")
RESULTS_DIR = os.path.join('..', 'results')
# SEEDS = (12, 34, 56, 78, 90)
SEEDS = (12, 34)

%matplotlib inline

In [44]:
plt.style.use('fivethirtyeight')
params = {
    'figure.figsize': (15, 8),
    'font.size': 24,
    'legend.fontsize': 20,
    'axes.titlesize': 28,
    'axes.labelsize': 24,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20
}
pylab.rcParams.update(params)
np.set_printoptions(suppress=True)

In [ ]:
torch.cuda.is_available()

True

In [ ]:
from exploration_strategies import EGreedyStrategy, EGreedyLinearStrategy, EGreedyExpStrategy, SoftMaxStrategy, GreedyStrategy
from networks import FCQ
from buffers import ReplayBuffer
from dnq import DQN
from utils import get_make_env_fn


In [ ]:
import importlib

# Use module names so each reload target is guaranteed to be a module object.
_submodule_names = [
    "exploration_strategies.epsilon_greedy_strategy",
    "exploration_strategies.epsilon_greedy_linear_strategy",
    "exploration_strategies.epsilon_greedy_exponential_strategy",
    "exploration_strategies.softmax_strategy",
    "exploration_strategies.greedy_strategy",
    "networks.fcq",
    "buffers.replay_buffer",
    "dnq.dnq",
    "utils.get_make_env_fn",
]

for _name in _submodule_names:
    _module = importlib.import_module(_name)
    importlib.reload(_module)

_package_names = ["exploration_strategies", "networks", "buffers", "dnq", "utils"]
for _name in _package_names:
    _pkg = importlib.import_module(_name)
    importlib.reload(_pkg)

from exploration_strategies import EGreedyStrategy, EGreedyLinearStrategy, EGreedyExpStrategy, SoftMaxStrategy, GreedyStrategy
from networks import FCQ
from buffers import ReplayBuffer
from dnq import DQN
from utils import get_make_env_fn

print("Reloaded packages and submodules with importlib.reload")

Reloaded packages and submodules with importlib.reload


In [ ]:
dqn_results = []
best_agent, best_eval_score = None, float('-inf')
for seed in SEEDS:
    environment_settings = {
        'env_name': 'CartPole-v1',
        'gamma': 1.00,
        'max_minutes': 20,
        'max_episodes': 10000,
        'goal_mean_100_reward': 475
    }
    
    value_model_fn = lambda nS, nA: FCQ(nS, nA, hidden_dims=(512,128))
    value_optimizer_fn = lambda net, lr: optim.RMSprop(net.parameters(), lr=lr)
    value_optimizer_lr = 0.0005

    # training_strategy_fn = lambda: EGreedyStrategy(epsilon=0.5)
    # training_strategy_fn = lambda: EGreedyLinearStrategy(init_epsilon=1.0,
    #                                                      min_epsilon=0.3, 
    #                                                      max_steps=20000)
    # training_strategy_fn = lambda: SoftMaxStrategy(init_temp=1.0, 
    #                                                min_temp=0.1, 
    #                                                exploration_ratio=0.8, 
    #                                                max_steps=20000)
    training_strategy_fn = lambda: EGreedyExpStrategy(init_epsilon=1.0,  
                                                      min_epsilon=0.3, 
                                                      decay_steps=20000)
    evaluation_strategy_fn = lambda: GreedyStrategy()

    replay_buffer_fn = lambda: ReplayBuffer(max_size=50000, batch_size=64)
    n_warmup_batches = 5
    update_target_every_steps = 10

    env_name, gamma, max_minutes, \
    max_episodes, goal_mean_100_reward = environment_settings.values()
    agent = DQN(replay_buffer_fn,
                value_model_fn,
                value_optimizer_fn,
                value_optimizer_lr,
                training_strategy_fn,
                evaluation_strategy_fn,
                n_warmup_batches,
                update_target_every_steps)

    make_env_fn, make_env_kargs = get_make_env_fn(env_name=env_name)
    result, final_eval_score, training_time, wallclock_time = agent.train(
        make_env_fn, make_env_kargs, seed, gamma, max_minutes, max_episodes, goal_mean_100_reward)
    dqn_results.append(result)
    if final_eval_score > best_eval_score:
        best_eval_score = final_eval_score
        best_agent = agent
dqn_results = np.array(dqn_results)
_ = BEEP()

el 00:00:00, ep 0000, ts 000015, ar 10 015.0±000.0, 100 015.0±000.0, ex 100 0.4±0.0, ev 028.0±000.0
el 00:01:01, ep 0135, ts 010565, ar 10 227.4±080.8, 100 096.2±080.2, ex 100 0.3±0.1, ev 266.6±115.3
el 00:02:02, ep 0185, ts 024803, ar 10 435.0±090.7, 100 213.5±130.1, ex 100 0.2±0.1, ev 324.2±129.0
el 00:03:03, ep 0218, ts 038947, ar 10 428.1±152.6, 100 318.4±149.0, ex 100 0.2±0.0, ev 419.1±108.0
el 00:04:04, ep 0248, ts 052411, ar 10 463.9±073.7, 100 392.2±138.6, ex 100 0.2±0.0, ev 463.6±077.3
el 00:04:23, ep 0258, ts 056578, ar 10 416.7±150.7, 100 409.9±133.6, ex 100 0.2±0.1, ev 477.1±063.2
--> reached_goal_mean_reward ✓
Training complete.
Final evaluation score 499.72±2.79 in 232.33s training time, 281.84s wall-clock time.



AttributeError: module 'numpy' has no attribute 'int'.
`np.int` was a deprecated alias for the builtin `int`. To avoid this error in existing code, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations